In [ ]:
import numpy as np
from typing import List, Tuple

class PDHGSolver:
    """
    @author: vohuynhquangnguyen
    Primal-Dual Hybrid Gradient (PDHG) solver for linear problems.
    """
    def __init__(self, A: np.ndarray, b: np.ndarray, c: np.ndarray, num_iterations: int) -> None:
        self.A = A
        self.b = b
        self.c = c

        self.n_primal = A.shape[1]
        self.n_dual = A.shape[0]
        
        self.tol = 1e-6
        self.theta = 1.0
        
        self.num_iterations = num_iterations
        self.x_iterates: List[np.ndarray] = []
    
    @staticmethod
    def _project_dual(mu: np.ndarray) -> np.ndarray:
        """
        Helper method to project dual variables to be non-negative.
        """
        return np.maximum(mu, 0)
    
    @staticmethod
    def _project_primal(x: np.ndarray) -> np.ndarray:
        """
        Helper method to project primal variables to be non-negative.
        """
        return np.maximum(x, 0)
    
    def _compute_stepsize(self):
        """
        Helper method to compute primal and dual steps.
        """
        maximum_step = 1 / np.linalg.norm(self.A, ord = 2)
        primal_step = maximum_step
        dual_step = maximum_step
        return primal_step, dual_step    

    def solve(self) -> Tuple[np.ndarray, np.ndarray]:
        """
        Run the PDHG iterations and return final and averaged solutions.
        """
        x = np.zeros(self.n_primal)
        mu = np.zeros(self.n_dual)
        x_bar = x.copy()

        primal_step, dual_step = self._compute_stepsize()

        for k in range(self.num_iterations):

            # --- Dual update using extrapolated x_bar ---
            mu_tilde = mu + dual_step * (self.A @ x_bar - self.b)
            mu_next = self._project_dual(mu_tilde)

            # --- Primal update ---
            x_grad = x - primal_step * (self.A.T @ mu_next + self.c)
            x_next = self._project_primal(x_grad)

            # --- Extrapolation for next x_bar ---
            x_bar = x_next + self.theta * (x_next - x)

            # --- Check convergence (primal residual) ---
            if np.linalg.norm(x_next - x) < self.tol:
                print(f"Terminated after {k} iterations.")
                break

            self.x_iterates.append(x_next.copy())
            x = x_next
            mu = mu_next

        x_avg = np.mean(self.x_iterates, axis=0)
        return x_bar, x_avg

In [33]:
import numpy as np
from typing import List, Tuple

class PDHGSolver:
    """
    @author: vohuynhquangnguyen
    Primal-Dual Hybrid Gradient (PDHG) solver for linear problems with Nesterov acceleration.
    """
    def __init__(self, A: np.ndarray, b: np.ndarray, c: np.ndarray, num_iterations: int) -> None:
        self.A = A
        self.b = b
        self.c = c
        self.n_primal = A.shape[1]
        self.n_dual = A.shape[0]
        self.num_iterations = num_iterations
        self.tol = 1e-6
        self.x_iterates: List[np.ndarray] = []
    
    @staticmethod
    def _project_dual(mu: np.ndarray) -> np.ndarray:
        """Project dual variables to be non-negative."""
        return np.maximum(mu, 0)
    
    @staticmethod
    def _project_primal(x: np.ndarray) -> np.ndarray:
        """Project primal variables to be non-negative."""
        return np.maximum(x, 0)
    
    def _compute_stepsize(self) -> Tuple[float, float]:
        """Compute step sizes for primal and dual variables with Nesterov adjustment."""
        norm_A = np.linalg.norm(self.A, ord=2)  # Spectral norm of A
        tau = 1 / (np.sqrt(2) * norm_A)  # Adjusted for theta ≤ 1
        sigma = 1 / (np.sqrt(2) * norm_A)
        return tau, sigma
    
    def solve(self) -> Tuple[np.ndarray, np.ndarray]:
        """Run PDHG iterations with Nesterov acceleration and return solutions."""
        x = np.zeros(self.n_primal)
        mu = np.zeros(self.n_dual)
        x_bar = x.copy()  # Extrapolated primal variable
        theta_prev = 1.0  # Initial theta for Nesterov sequence
        
        tau, sigma = self._compute_stepsize()
        
        for k in range(self.num_iterations):
            # --- Dual update using extrapolated x_bar ---
            mu_tilde = mu + sigma * (self.A @ x_bar - self.b)
            mu_next = self._project_dual(mu_tilde)
            
            # --- Primal update ---
            x_grad = x - tau * (self.A.T @ mu_next + self.c)
            x_next = self._project_primal(x_grad)
            
            # --- Nesterov momentum update for theta ---
            theta = (1 + np.sqrt(1 + 4 * theta_prev**2)) / 2  # Nesterov formula
            beta = (theta_prev - 1) / theta  # Momentum parameter
            
            # --- Extrapolation with momentum ---
            x_bar = x_next + beta * (x_next - x)
            theta_prev = theta  # Track theta for next iteration
            
            # --- Check convergence ---
            if np.linalg.norm(x_next - x) < self.tol:
                print(f"Converged after {k} iterations.")
                break
            
            self.x_iterates.append(x_next.copy())
            x = x_next
            mu = mu_next
        
        x_avg = np.mean(self.x_iterates, axis=0)
        return x, x_avg

In [ ]:
import numpy as np
from scipy.io import mmread, mmwrite

# --- Problem parameters ---
A = np.array([
    [6, 3, 1, 0, 0],
    [3, -1, 0, 1, 0],
    [1, 1/4, 0, 0, 1]])
np.savetxt('A.csv', A, delimiter=',')
mmwrite('A.mtx', A)

b = np.array([40, 0, 4])
np.savetxt('b.csv', b, delimiter=',')

c = np.array([-30, -10, 0, 0, 0])
np.savetxt('c.csv', c, delimiter=',')

K = 100000
solver = PDHGSolver(A, b, c, K)
x_final, x_avg = solver.solve()

print(f"Optimal solution: x1 = {x_final[0]:.4f}, x2 = {x_final[1]:.4f}")
print(f"Objective value: {-np.dot(c, x_final):.2f}")

Terminated after 8048 iterations.
Optimal solution: x1 = 1.3677, x2 = 10.5970
Objective value: 147.00


In [3]:
import torch
import numpy as np
from typing import List, Tuple

class PowerIteration:
    """
    @author: Vo, Huynh Quang Nguyen
    """
    def __init__(self, A: np.ndarray, x_init: np.ndarray, num_iterations: int, tol: float):
        self.A = A
        self.A_trans = A.T
        self.x = x_init
        self.num_iterations = num_iterations
        self.tol = tol

    def solve(self):
        # --- Normalize the initial vector ---
        v_k = self.x.astype(float)
        v_norm = np.linalg.norm(v_k)

        if v_norm == 0:
            raise ValueError("Initial vector has zero norm! Cannot normalize!")
        v_k /= v_norm

        for _ in range(self.num_iterations):

            # w_k = A @ v_k
            w_k = torch.matmul(torch.from_numpy(self.A), 
                               torch.from_numpy(v_k))
            w_k = w_k.numpy()
            norm_w_k = np.linalg.norm(w_k)
            if norm_w_k < self.tol:
                break
            w_k /= norm_w_k

            # v_{k+1} = A.T @ w_k
            v_next = torch.matmul(torch.from_numpy(self.A_trans),
                                  torch.from_numpy(w_k))
            v_next = v_next.numpy()
            norm_v_next = np.linalg.norm(v_next)

            if norm_v_next < self.tol:
                break
            v_k = v_next / norm_v_next

        # Approximate the spectral norm
        w_final = torch.matmul(torch.from_numpy(self.A), 
                               torch.from_numpy(v_k))
        w_final = w_final.numpy()
        spectral_norm_est = np.linalg.norm(w_final)

        return spectral_norm_est